# Pelican file events: live EarthScope GNSS data

This notebook subscribes to a Pelican namespace, and plots each new
object as it appears. The data is EarthScope GNSS displacement:
east, north and up, one file per minute.

Nothing is downloaded. Each object is read straight from the
federation into memory, and the NDP Endpoint is not in the data path.

Everything here goes through `ndp-ep`. There is no STOMP, no
WebSocket handling and no `asyncio` to write.

In [ ]:
%pip install "ndp-ep[pelican]"

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

## Configuration

**Choose your own `CLIENT_ID`.** It identifies your subscriber. Two
clients sharing an id compete for the same events, so reusing someone
else's will take events away from them.

`EVENT_SOURCE` is the namespace to watch. The credentials are checked
by the event server against its own store; they are unrelated to the
Endpoint token, and will be replaced by an access token later.

In [ ]:
CLIENT_ID = "my-client"  # <- change this
EVENT_SOURCE = "osdf/vdc/public/pelican_protocol"

ENDPOINT_URL = "http://155.101.6.191:8003"
EVENT_USERNAME = "saleem"
EVENT_PASSWORD = "1234"

## Connect and subscribe

One client object covers both halves: the subscription that tells you
an object appeared, and the reads that fetch it.

In [ ]:
from ndp_ep import APIClient

client = APIClient(base_url=ENDPOINT_URL)

subscription = client.subscribe_pelican(
    EVENT_SOURCE,
    client_id=CLIENT_ID,
    username=EVENT_USERNAME,
    password=EVENT_PASSWORD,
)

subscription.wait_until_connected(timeout=30)
subscription.status

## Plot each file as it arrives

`for event in subscription` blocks until the next event and yields it
once. Redeliveries are suppressed against a record on disk, so no
bookkeeping is needed here and restarting the notebook will not
reprocess what it already plotted.

This is live data and the publisher writes roughly one file per
minute, so expect the cell below to take a few minutes. A new
subscriber starts from the moment it connects; it does not receive the
namespace's history. Raise `MAX_EVENTS` to keep it running longer, or
drop the counter entirely to run until interrupted.

In [ ]:
%pip install plotly anywidget pandas

In [ ]:
import io

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

MAX_POINTS = 600
MAX_EVENTS = 3

all_data = pd.DataFrame()


def create_figure(title, color):
    fig = go.FigureWidget()
    fig.add_scatter(mode="lines", line=dict(color=color, width=2),
                    name=title)
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=250,
        margin=dict(l=50, r=30, t=40, b=40),
        xaxis_title="Time",
        yaxis_title=title,
    )
    return fig


east_fig = create_figure("East", "royalblue")
north_fig = create_figure("North", "green")
up_fig = create_figure("Up", "firebrick")

display(east_fig)
display(north_fig)
display(up_fig)

for received, event in enumerate(subscription, start=1):

    print(f"Received {event.name}")

    raw = client.pelican_read(event.url)

    df = pd.read_csv(io.BytesIO(raw))
    df["datetime"] = pd.to_datetime(df["time"], unit="ms", utc=True)

    all_data = pd.concat([all_data, df], ignore_index=True)
    if len(all_data) > MAX_POINTS:
        all_data = all_data.iloc[-MAX_POINTS:].copy()

    x = all_data["datetime"]
    for figure, column in (
        (east_fig, "east"),
        (north_fig, "north"),
        (up_fig, "up"),
    ):
        with figure.batch_update():
            figure.data[0].x = x
            figure.data[0].y = all_data[column]

    if received >= MAX_EVENTS:
        break

## All three components on one figure

In [ ]:
combined = go.FigureWidget()

for column, color in (("east", "royalblue"),
                     ("north", "green"),
                     ("up", "firebrick")):
    combined.add_scatter(
        x=all_data["datetime"],
        y=all_data[column],
        name=column.capitalize(),
        line=dict(color=color, width=2),
    )

combined.update_layout(template="plotly_white", height=350,
                       xaxis_title="Time")
display(combined)

## Browsing without subscribing

The namespace can be listed and read directly, with no subscription
involved. References are accepted in any of the spellings that turn
up in practice: a bare path, `osdf://...`, or `pelican://host/...`.

In [ ]:
objects = client.pelican_list(EVENT_SOURCE)

print(f"{len(objects)} objects")
objects[-3:]

In [ ]:
raw = client.pelican_read(objects[-1])

print(raw[:200].decode())

## Downloading to disk

`pelican_read` keeps the object in memory. Use `pelican_fetch` when
a file on disk is what you want; an existing target is an error
rather than a silent overwrite.

In [ ]:
path = client.pelican_fetch(objects[-1], "./data/")

print(path, path.stat().st_size, "bytes")

## The raw events

Each event carries the object's name, a reference ready to pass to
`pelican_read`, its size, and the modification time reported by the
server.

Passing a `timeout` bounds the wait, so the cell ends even if the
publisher goes quiet.

In [ ]:
for received, event in enumerate(subscription.events(timeout=120), 1):
    print(event.name, event.size, event.mod_time)
    print("   ", event.url)
    if received >= 2:
        break

## Closing

Closing performs the WebSocket closing handshake, so the event server
does not sit on a half-open connection. Using the subscription as a
context manager (`with client.subscribe_pelican(...) as subscription:`)
does this automatically.

In [ ]:
subscription.close()
subscription.status["metrics"]